In [3]:
from dotenv import dotenv_values
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance
from openai import OpenAI

In [4]:
env = dotenv_values("../../../.env")

In [5]:
EMBEDDING_DIM = 1536
EMBEDDING_MODEL = "text-embedding-3-small"

def get_openai_client():
    return OpenAI(api_key=env["OPENAI_API_KEY"])

def get_embedding(text):
    openai_client = get_openai_client()
    result = openai_client.embeddings.create(
        input=[text],
        model=EMBEDDING_MODEL,
        dimensions=EMBEDDING_DIM
    )

    return result.data[0].embedding

In [6]:
qdrant_client = QdrantClient(":memory:")

In [7]:
books = [
    {
        "name": "The Great Gatsby",
        "author": "F. Scott Fitzgerald",
        "description": "A novel set in the Roaring Twenties, exploring themes of wealth, love, and the American Dream.",
        "year": 1925
    },
    {
        "name": "To Kill a Mockingbird",
        "author": "Harper Lee",
        "description": "A gripping tale of racial injustice and childhood innocence in the American South.",
        "year": 1960
    }
]

In [8]:
QDRAND_COLLECTION_NAME = "books"

if not qdrant_client.collection_exists(collection_name=QDRAND_COLLECTION_NAME):
    print("Tworzę kolekcję.")
    qdrant_client.create_collection(
        collection_name=QDRAND_COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE)
    )

Tworzę kolekcję.


In [9]:
qdrant_client.upsert(
    collection_name=QDRAND_COLLECTION_NAME,
    points=[
        PointStruct(
            id=idx,
            vector=get_embedding(f'{book["name"]} {book["description"]} autorstwa: {book["author"]}'),
            payload=book
        )
        for idx, book in enumerate(books)
    ]
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
from qdrant_client.http.models import NearestQuery

Note: you may need to restart the kernel to use updated packages.


In [ ]:
results = qdrant_client.query_points(
    collection_name=QDRAND_COLLECTION_NAME,
    query=NearestQuery(
        nearest=get_embedding("A novel about love and wealth in the 1920s")
    ),
    limit=3
)

for result in results.points:
    print("Tytuł:", result.payload["name"], "OPIS", result.payload["description"], "WYNIK", result.score)

Tytuł: The Great Gatsby OPIS A novel set in the Roaring Twenties, exploring themes of wealth, love, and the American Dream. WYNIK 0.573318094111477
Tytuł: To Kill a Mockingbird OPIS A gripping tale of racial injustice and childhood innocence in the American South. WYNIK 0.2038279718654247


In [14]:
def assure_db_collection_exists(
        qdrant_client,
        collection_name,
        embedding_dim
):
    if not qdrant_client.collection_exists(collection_name):
        print(f"Tworzę kolekcję {collection_name}.")
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=embedding_dim,
                distance=Distance.COSINE
                )
        )
    else:
        print(f"Kolekcja {collection_name} już istnieje.")

In [15]:
def add_note_to_db(
        qdrant_client,
        collection_name,
        text
):
    points_count = qdrant_client.count(
        collection_name=collection_name,
        exact=True
        )

    qdrant_client.upsert(
        collection_name=collection_name,
        points=[
            PointStruct(
                id=points_count.count + 1,
                vector=get_embedding(text=text),
                payload=
                {
                    "text": text
                 }
            )
            
        ]
    )

In [26]:
def list_notes_from_db(
        qdrant_client,
        collection_name,
        query=None
):
    if not query:
        notes = qdrant_client.scroll(collection_name=collection_name, limit=10)[0]
        result = []
        for note in notes:
            result.append({
                "text": note.payload["text"],
                "score" : None
            })
        return result

    else:
        notes = qdrant_client.query_points(
            collection_name=collection_name,
            #query_vector=get_embedding(text=query),
            query=get_embedding(text=query),
            limit=10
        ).points
        result = []
        for note in notes:
            result.append({
                "text": note.payload["text"],
                "score" : note.score
            })
        return result

In [27]:
LUNCH_NOTES_COLLECTION_NAME = "lunch_notes"

assure_db_collection_exists(
    qdrant_client=qdrant_client,
    collection_name=LUNCH_NOTES_COLLECTION_NAME,
    embedding_dim=EMBEDDING_DIM
)

add_note_to_db(
    qdrant_client=qdrant_client,
    collection_name=LUNCH_NOTES_COLLECTION_NAME,
    text="Note about today's lunch"
)

add_note_to_db(
    qdrant_client=qdrant_client,
    collection_name=LUNCH_NOTES_COLLECTION_NAME,
    text="Note about yesterday's lunch"
)

add_note_to_db(
    qdrant_client=qdrant_client,
    collection_name=LUNCH_NOTES_COLLECTION_NAME,
    text="Note about the day before yesterday's lunch"
)

Kolekcja lunch_notes już istnieje.


In [28]:
list_notes_from_db(qdrant_client=qdrant_client, collection_name=LUNCH_NOTES_COLLECTION_NAME)

[{'text': "Note about today's lunch", 'score': None},
 {'text': "Note about yesterday's lunch", 'score': None},
 {'text': "Note about the day before yesterday's lunch", 'score': None},
 {'text': "Note about today's lunch", 'score': None},
 {'text': "Note about yesterday's lunch", 'score': None},
 {'text': "Note about the day before yesterday's lunch", 'score': None},
 {'text': "Note about today's lunch", 'score': None},
 {'text': "Note about yesterday's lunch", 'score': None},
 {'text': "Note about the day before yesterday's lunch", 'score': None}]

In [29]:
list_notes_from_db(
    qdrant_client=qdrant_client,
    collection_name=LUNCH_NOTES_COLLECTION_NAME,
    query="yesterday"
    )

[{'text': "Note about the day before yesterday's lunch",
  'score': 0.4811618080520449},
 {'text': "Note about the day before yesterday's lunch",
  'score': 0.4811618080520449},
 {'text': "Note about the day before yesterday's lunch",
  'score': 0.4811618080520449},
 {'text': "Note about yesterday's lunch", 'score': 0.460442974234243},
 {'text': "Note about yesterday's lunch", 'score': 0.460442974234243},
 {'text': "Note about yesterday's lunch", 'score': 0.460442974234243},
 {'text': "Note about today's lunch", 'score': 0.3114295501252248},
 {'text': "Note about today's lunch", 'score': 0.3114295501252248},
 {'text': "Note about today's lunch", 'score': 0.3114295501252248}]